# 10 — HyperTempNet: il modello novel della tesi (ipergrafo TEMPORALE)
**Contributo**: gli ipergrafi di connettività **spaziale** (canali) NON aiutano l'imagined speech
— la connettività codifica il **soggetto** (NMI 0.95), non la **parola** (NMI 0.001). Proponiamo un
ipergrafo sui **SEGMENTI TEMPORALI** (dinamiche della parola), ispirato a Hyper-MML (Kang et al. 2026).

`raw → conv multi-scala (16/32/64/128) → conv spaziale → K=10 segmenti (nodi) → ipergrafo appreso (HGNN) → clf`

**Risultato (5 seed, subject-dependent)**: 0.701 vs baseline 0.55 (Wilcoxon p=0.0004); l'ipergrafo
temporale aggiunge +0.046 (p=0.0012). Ablation `temporal_hg=False` = senza ipergrafo.

> Env `daniele_311` con GPU. Ogni run ~alcuni minuti su GPU.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
import track3_config as C, track3_train as T
print(C.summary()); assert C.DATA_ROOT is not None, C._no_data_msg()

## §1 — HyperTempNet subject-dependent: ipergrafo ON vs OFF (ablation)
Il confronto che isola il contributo dell'ipergrafo temporale (stesso modello, cambia solo l'ipergrafo).

In [ ]:
TK = dict(epochs=200, patience=30, lr=1e-3, batch_size=32, label_smoothing=0.1)
df_on, res_on = T.run_subject_dependent('hypertempnet', pp_kwargs=C.PP_MINIMAL,
    model_kwargs=dict(temporal_hg=True, K_seg=10, n_edges=16), train_kwargs=TK)
df_off, res_off = T.run_subject_dependent('hypertempnet', pp_kwargs=C.PP_MINIMAL,
    model_kwargs=dict(temporal_hg=False, K_seg=10, n_edges=16), train_kwargs=TK)
T.save_metrics(df_on, 'hypertempnet_on'); T.save_metrics(df_off, 'hypertempnet_off')
print(f"ipergrafo ON : {df_on.test_acc.mean():.3f} ± {df_on.test_acc.std():.3f}")
print(f"ipergrafo OFF: {df_off.test_acc.mean():.3f} ± {df_off.test_acc.std():.3f}")
w = wilcoxon(df_on.test_acc.values, df_off.test_acc.values)
print(f"Wilcoxon ON vs OFF: p={w.pvalue:.4f}  ({int((df_on.test_acc.values>df_off.test_acc.values).sum())}/15 meglio con ipergrafo)")

## §2 — Confronto col baseline (Shallow) sui soggetti
Shallow ri-addestrata nella stessa pipeline per un confronto equo.

In [ ]:
df_sh, _ = T.run_subject_dependent('shallow', pp_kwargs=C.PP_MINIMAL, train_kwargs=TK)
comp = pd.DataFrame({'HyperTempNet': df_on.test_acc, 'HT-ablation': df_off.test_acc, 'Shallow': df_sh.test_acc})
print(comp.mean().round(3).to_string())
w2 = wilcoxon(df_on.test_acc.values, df_sh.test_acc.values)
print(f"\nHyperTempNet vs Shallow: Δ={df_on.test_acc.mean()-df_sh.test_acc.mean():+.3f}  Wilcoxon p={w2.pvalue:.4f}")
fig, ax = plt.subplots(figsize=(9,4))
comp.plot(kind='bar', ax=ax); ax.axhline(C.CHANCE_LEVEL, color='k', ls='--', lw=1, label='chance')
ax.set_xlabel('soggetto'); ax.set_ylabel('test acc'); ax.set_title('HyperTempNet vs ablation vs Shallow')
plt.tight_layout(); plt.savefig(C.FIG_DIR/'hypertempnet_per_subject.png', dpi=130); plt.show()

## §3 — Confusion matrix + curve


In [ ]:
T.plot_confusion(res_on, model_name='hypertempnet'); plt.show()
T.plot_training_curves(res_on, subject=1); plt.show()

## §4 — Controllo dual (temporale + spaziale in parallelo)
Verifica che l'ipergrafo *spaziale* non aggiunga sopra il temporale (spaziale da solo ≈ chance).

In [ ]:
df_ts, _ = T.run_subject_dependent('hyperdualnet', pp_kwargs=C.PP_MINIMAL,
    model_kwargs=dict(use_temporal=True, use_spatial=True), train_kwargs=TK)
df_s, _  = T.run_subject_dependent('hyperdualnet', pp_kwargs=C.PP_MINIMAL,
    model_kwargs=dict(use_temporal=False, use_spatial=True), train_kwargs=TK)
print(f"spaziale-solo : {df_s.test_acc.mean():.3f}  (atteso ~chance)")
print(f"dual (T+S)    : {df_ts.test_acc.mean():.3f}  vs temporale-solo {df_on.test_acc.mean():.3f}")

## §5 — HyperTempNet vs Shallow sui 3 protocolli
Confronto completo: subject-dependent, subject-mixed, subject-independent (holdout).

In [ ]:
def _agg(df, proto):
    return (df['test_acc'].mean() if proto=='dependent'
            else df.loc['ALL','test_acc'] if proto=='mixed' else df.iloc[0]['test_acc'])
res3 = {}
for name, mk in [('hypertempnet', dict(temporal_hg=True, K_seg=10, n_edges=16)), ('shallow', {})]:
    dd,_ = T.run_subject_dependent(name, pp_kwargs=C.PP_MINIMAL, model_kwargs=mk, train_kwargs=TK, verbose=False)
    dm,_ = T.run_subject_mixed(name, pp_kwargs=C.PP_MINIMAL, model_kwargs=mk,
        train_kwargs=dict(**{**TK, 'batch_size':64}), verbose=False)
    di,_ = T.run_subject_independent(name, mode='holdout', pp_kwargs=C.PP_MINIMAL, model_kwargs=mk,
        train_kwargs=dict(**{**TK, 'batch_size':64}), verbose=False)
    res3[name] = {'dependent':_agg(dd,'dependent'), 'mixed':_agg(dm,'mixed'), 'independent':_agg(di,'independent')}
tab3 = pd.DataFrame(res3).T[['dependent','mixed','independent']].round(3)
print('HyperTempNet vs Shallow, 3 protocolli (chance 0.20):'); print(tab3.to_string())
print('\nΔ (HyperTempNet - Shallow):'); print((tab3.loc['hypertempnet']-tab3.loc['shallow']).round(3).to_string())
tab3.to_csv(C.RESULTS_DIR/'hypertempnet_vs_shallow_3protocols.csv')

## Conclusioni
- **HyperTempNet** batte il baseline (p<0.001) e l'**ipergrafo temporale** contribuisce significativamente (p<0.01).
- L'ipergrafo **spaziale** resta a chance da solo e non aggiunge nel dual → conferma: la connettività
  codifica il soggetto, non la parola.
- **Novelty**: ipergrafo temporale (segmenti) per imagined speech, adattato da Hyper-MML (Kang et al.
  2026) + costruzione dinamica (DHSLP) + front-end multi-scala (EEG-Inception).

Dettagli e numeri finali (5 seed) in `RESULTS.md`.